In [ ]:
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(SeuratDisk)
library(cowplot)

In [ ]:
sc <-readRDS('data/spatial/20230911_tonsil_atlas_rna_seurat_obj.rds')
donor = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
sc <- subset(sc, subset = donor_id %in% donor)
sc[["percent.mt"]] <- PercentageFeatureSet(sc, pattern = "^MT-")
glimpse(sc)

In [ ]:
donor_levels <- levels(factor(sc@meta.data$donor_id))
donor_colors <- c('cyan3', 'darkgoldenrod1', 'deeppink', 'chartreuse3', 'dodgerblue', 'orange')
cnames <- setNames(donor_colors, donor_levels)
VlnPlot(sc, features = 'nCount_RNA', layer='counts', group.by='donor_id',raster=FALSE,alpha=0.2) + scale_fill_manual(values=cnames) 

In [ ]:
ggplot(sc@meta.data, aes(x=nCount_RNA, color=donor_id, fill=donor_id)) +
  geom_density(alpha=0.2) +
  theme_classic() +
  scale_x_log10()

In [ ]:
VlnPlot(sc, features = 'nFeature_RNA', group.by='donor_id') + scale_fill_manual(values=cnames) 

In [ ]:
ggplot(sc@meta.data, aes(x=nFeature_RNA,fill=donor_id)) +
   geom_density(alpha = 0.2) + 
    theme_classic() +
    scale_x_log10()

In [ ]:
VlnPlot(sc, features = 'percent.mt', group.by='donor_id') +
  scale_fill_manual(values=cnames) +
  geom_hline(yintercept=10,color='red')


In [ ]:
ggplot(sc@meta.data, aes(x=percent.mt,fill=donor_id)) +
   geom_density(alpha = 0.2) + 
  scale_x_log10()+
    theme_classic()  


In [ ]:
QC_plots <- plot_grid(
  FeatureScatter(sc, feature1 = "nCount_RNA", feature2 = "nFeature_RNA", 
                 group.by = "donor_id", pt.size = 0.5),
  FeatureScatter(sc, feature1 = "nCount_RNA", feature2 = "percent.mt", 
                 group.by = "donor_id", pt.size = 0.5),
  FeatureScatter(sc, feature1 = "nFeature_RNA", feature2 = "percent.mt", 
                 group.by = "donor_id", pt.size = 0.5),
  ncol = 3, 
  labels = c("A", "B", "C")
)

ggsave("QC_scatter_plots.pdf", QC_plots, width = 18, height = 6)

In [ ]:

p1 <- ggplot(sc@meta.data, aes(x=nCount_RNA, color=donor_id, fill=donor_id)) + 
  geom_density(alpha=0.2) + 
  theme_classic() + 
  scale_x_log10() + 
  labs(x = "nCount_RNA", y = "Density")

p2 <- ggplot(sc@meta.data, aes(x=nFeature_RNA, fill=donor_id)) + 
  geom_density(alpha=0.2) + 
  theme_classic() + 
  scale_x_log10() + 
  labs(x = "nFeature_RNA", y = "Density")

p3 <- ggplot(sc@meta.data, aes(x=percent.mt, fill=donor_id)) + 
  geom_density(alpha=0.2) + 
  scale_x_log10() + 
  theme_classic() +
  labs(x = "Percent Mitochondrial", y = "Density")

QC_density_plots <- plot_grid(p1, p2, p3, 
                               ncol = 3, 
                               labels = c("A", "B", "C"),
                               align = 'h')

ggsave("QC_density_plots.pdf", QC_density_plots, width = 18, height = 5)